## 0. Configurando sessão spark

In [8]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver_aluno")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

# Projeto usado para faturamento das consultas
spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

In [9]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [10]:
from pyspark.sql import functions as F

## 2. Geração de parâmetros

In [11]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_silver_uf = f"{par_source_project}.silver.uf"

par_source_gold_fato_indicador_uf = f"{par_source_project}.gold.fato_indicador_uf"

## 3. Leitura dos dados da origem

In [12]:
df_scr_uf = spark.read.format("bigquery").option("table",par_source_silver_uf).load()

## 4. Transformações

In [13]:
fato_ind_uf = (
    df_scr_uf
    .select("ano","sigla_uf","rede_id","serie",
            "taxa_alfabetizacao","media_portugues",
            *[f"proporcao_aluno_nivel_{i}" for i in range(9)])
)

## 5. Armazenamento no BQ

In [15]:
(
    fato_ind_uf.write.format("bigquery")
    .option("table", par_source_gold_fato_indicador_uf)
    .option("writeMethod", "direct")
    .mode("overwrite")
    .save()
)